# Homework 10: Improving RAG with Reranking, Chunking, and Conversational Memory
Dataset: `ccdv/arxiv-summarization`, document subset, first 100 train documents. Vector DB: ChromaDB.

In [1]:
# %pip install -q chromadb sentence-transformers datasets pandas

## Preprocessing
Remove the `@xmath` and `@xcite` placeholders and fix spacing around punctuation.

In [2]:
import re, numpy as np, pandas as pd, chromadb
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder

# first shard of the document subset holds the first rows of the train split
ds = load_dataset("ccdv/arxiv-summarization", data_files={"train": "document/train-00000-of-00015.parquet"}, split="train[:100]")

def clean(t):
    t = re.sub(r"@xmath\d+|@xcite", " ", t)
    t = re.sub(r"\s+([.,;:?!)\]])", r"\1", t)
    t = re.sub(r"([(\[])\s+", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

docs = [clean(a) for a in ds["article"]]
print(len(docs), "documents,", sum(len(d.split()) for d in docs), "words")

/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1109 22:27:59.196000 38048 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


100 documents, 522033 words


## Chunking
Strategy 1: fixed windows of 150 words with 30 word overlap. Strategy 2: whole sentences grouped up to 150 words.

In [3]:
def fixed_chunks(text, size=150, overlap=30):
    w = text.split()
    return [" ".join(w[i:i + size]) for i in range(0, max(len(w) - overlap, 1), size - overlap)]

def sentence_chunks(text, max_words=150):
    chunks, cur, n = [], [], 0
    for s in re.split(r"(?<=[.!?])\s+", text):
        k = len(s.split())
        if cur and n + k > max_words:
            chunks.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += k
    return chunks + [" ".join(cur)] if cur else chunks

store = {name: [(i, c) for i, d in enumerate(docs) for c in fn(d)]
         for name, fn in {"fixed": fixed_chunks, "sentence": sentence_chunks}.items()}
for name, ch in store.items():
    print(name, len(ch), "chunks, mean", round(np.mean([len(c.split()) for _, c in ch])), "words")

fixed 4373 chunks, mean 149 words
sentence 3848 chunks, mean 136 words


## Embedding and indexing in ChromaDB

In [4]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
client = chromadb.EphemeralClient()
cols = {}
for name, ch in store.items():
    col = client.create_collection(name, configuration={"hnsw": {"space": "cosine"}})
    texts = [c for _, c in ch]
    emb = embedder.encode(texts, batch_size=64, normalize_embeddings=True)
    for s in range(0, len(texts), 2000):
        e = min(s + 2000, len(texts))
        col.add(ids=[f"{name}{j}" for j in range(s, e)], documents=texts[s:e],
                embeddings=emb[s:e].tolist(), metadatas=[{"doc": ch[j][0]} for j in range(s, e)])
    cols[name] = col
    print(name, col.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9992.44it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7810.76it/s]

fixed 4373


sentence 3848


## Retrieval without and with reranking
Reranking takes the top 20 vector hits and reorders them with a cross encoder.

In [5]:
def search(query, strategy, rerank=False, k=5, n=20):
    r = cols[strategy].query(query_embeddings=embedder.encode([query], normalize_embeddings=True).tolist(),
                             n_results=n if rerank else k)
    hits = [(m["doc"], t) for m, t in zip(r["metadatas"][0], r["documents"][0])]
    if rerank:
        scores = reranker.predict([(query, t) for _, t in hits])
        hits = [hits[i] for i in np.argsort(-scores)]
    return hits[:k]

def show(query, strategy, rerank):
    print(f"\n[{strategy}, rerank={rerank}] {query}")
    for d, t in search(query, strategy, rerank):
        print(f"  doc {d}: {t[:110]}")

queries = {"Q1": "Why does resistance grow linearly with magnetic field in topological insulators?",
           "Q2": "How is dark energy tested with cosmological observations?",
           "Q3": "How do groups of animals arrange themselves under attraction and repulsion?"}
for q in queries.values():
    for rr in (False, True):
        show(q, "sentence", rr)
for rr in (False, True):
    show(queries["Q1"], "fixed", rr)


[sentence, rerank=False] Why does resistance grow linearly with magnetic field in topological insulators?
  doc 20: abrikosov suggested a quantum - limit origin of lmr for the homogenous system with a gapless linear energy spe
  doc 20: when the level broadening is comparable to or larger than the landau - level separation and the conduction ele
  doc 20: we find that a positive, nonsaturating and dominantly linear magnetoresistance can appear within quite wide ma
  doc 20: is shown as a function of the magnetic field for different values of zero - magnetic - field mobility: (a), (b
  doc 71: the discovery of topological insulators in 2d and 3d has recently attracted a great deal of attention (referen

[sentence, rerank=True] Why does resistance grow linearly with magnetic field in topological insulators?
  doc 20: we find that a positive, nonsaturating and dominantly linear magnetoresistance can appear within quite wide ma
  doc 20: when the level broadening is comparable to or large

  doc 26: the current observations, such as sneia (supernovae type ia), cmb (cosmic microwave background) and large scal
  doc 72: in the past decade, observations of type ia supernovae (sne ia; riess et al. 1998; schmidt et al. 1998; perlmu
  doc 72: bertolami & silva (2006) considered the use of grbs at calibrated with the bursts at as distance markers to st
  doc 72: in order to constrain the cosmological parameters, we have applied the calibrated relations to grb data at hig
  doc 72: the best - fit values are. for a prior of, we obtain, which is consistent with the cosmological constant in a 

[sentence, rerank=True] How is dark energy tested with cosmological observations?
  doc 26: the current observations, such as sneia (supernovae type ia), cmb (cosmic microwave background) and large scal
  doc 72: more recently, schaefer (2007) used five grb relations calibrated with 69 grbs by assuming two adopted cosmolo
  doc 72: the 42 grb data at are obtained by utilizing the five relati

  doc 10: we will show below that for the quasi - two - dimensional model, this is not the case, and indeed, some minimi
  doc 10: this assumption is appropriate when drag forces dominate momentum, commonly known in fluid dynamics as the low
  doc 10: the first term on the right - hand side of ([eq: locusts]) describes endogenous forces; measures the force tha
  doc 10: biological aggregations such as fish schools, bird flocks, bacterial colonies, and insect swarms have characte
  doc 10: here, is the distance between particles, is the characteristic length scale of attraction, and is the characte

[fixed, rerank=False] Why does resistance grow linearly with magnetic field in topological insulators?
  doc 20: magnetoresistance appears within a quite wide range of magnetic field and persists up to room temperature. thi
  doc 20: (b), (c), (d), (e), and (f). the inset of (a) illustrates the same for a larger magnetic - field range. the fi
  doc 20: in topological insulator bi se nanoribb

## Conversational memory
The memory stores past user turns and prepends the last ones to a new query, so a follow up with a pronoun keeps its topic.

In [6]:
class Memory:
    def __init__(self, turns=1):
        self.history, self.turns = [], turns
    def rewrite(self, query):
        q = " ".join(self.history[-self.turns:] + [query])
        self.history.append(query)
        return q

T1 = "What elements pollute the atmosphere of cool white dwarfs?"
T2 = "How was its temperature determined?"
mem = Memory()
for t in (T1, T2):
    q = mem.rewrite(t)
    show(q, "sentence", True)
T2_mem = q
show(T2, "sentence", True)  # same follow up without memory


[sentence, rerank=True] What elements pollute the atmosphere of cool white dwarfs?


  doc 41: polluted white dwarfs (typed with a suffix `` z '') provide an opportunity to investigate the ultimate fate of
  doc 41: our model atmosphere analysis revealed the lowest iron to calcium abundance ratio of any cool polluted white d
  doc 41: a similar finding, but with a higher fraction of water, was found in the case of sdss j124231.07.6. the materi
  doc 41: this abundance ratio is slightly lower than that of nltt 43806 (=-0.2, *???. finally, the abundance ratio is a
  doc 41: an analysis of uv and optical spectra of two additional white dwarfs by show the accreting source to be of a r

[sentence, rerank=True] What elements pollute the atmosphere of cool white dwarfs? How was its temperature determined?


  doc 41: polluted white dwarfs (typed with a suffix `` z '') provide an opportunity to investigate the ultimate fate of
  doc 41: our model atmosphere analysis revealed the lowest iron to calcium abundance ratio of any cool polluted white d
  doc 41: we present a spectroscopic and photometric analysis of an hitherto unknown cool, polluted white dwarf (nltt 19
  doc 41: a similar finding, but with a higher fraction of water, was found in the case of sdss j124231.07.6. the materi
  doc 41: this abundance ratio is slightly lower than that of nltt 43806 (=-0.2, *???. finally, the abundance ratio is a

[sentence, rerank=True] How was its temperature determined?


  doc 22: the temperature of the water could be varied between and with a precision of. the temperature was brought to a
  doc 61: under these conditions, it may be non - trivial to distinguish observationally between a disk where the temper
  doc 22: [fig_temp_laser],width=325] the laser beam signal shown in fig. [fig_temp_laser] changes its polarity around, 
  doc 96: gliese 229b is not only the first brown dwarf recognized as genuine, but it is also the brightest and best - s
  doc 61: similarly, we selected a value of, in part because it produces a reasonable s - curve. however, we checked a r


## Evaluation
Relevance uses document level labels set manually (we read the abstracts and picked the documents that answer each query) and keyword matching at chunk level: a chunk is relevant if it comes from a chosen document and mentions a key term. Recall uses all relevant chunks in the collection as the denominator. K = 5.

In [7]:
qrels = {"Q1": ({20}, ["magnetoresistance"]),
         "Q2": ({26, 72}, ["dark energy", "equation of state"]),
         "Q3": ({10}, ["swarm", "aggregat", "attract", "repuls", "flock", "school"]),
         "T1": ({41}, ["pollut", "metal", "abundance"]),
         "T2": ({41}, ["temperature"])}
texts = {**queries, "T1": T1, "T2": T2}

def is_rel(qid, d, t):
    docs_rel, terms = qrels[qid]
    return d in docs_rel and any(x in t.lower() for x in terms)

def evaluate(qid, query, strategy, rerank, k=5):
    flags = [is_rel(qid, d, t) for d, t in search(query, strategy, rerank, k)]
    total = sum(is_rel(qid, d, t) for d, t in store[strategy])
    rr = next((1 / (i + 1) for i, f in enumerate(flags) if f), 0)
    return {"P@5": sum(flags) / k, "R@5": sum(flags) / total, "RR": rr}

rows = []
for strategy in store:
    for rerank in (False, True):
        for qid, q in texts.items():
            rows.append({"query": qid, "strategy": strategy, "rerank": rerank, "memory": False, **evaluate(qid, q, strategy, rerank)})
        rows.append({"query": "T2", "strategy": strategy, "rerank": rerank, "memory": True,
                     **evaluate("T2", T2_mem, strategy, rerank)})
df = pd.DataFrame(rows).round(3)
df

,query,strategy,rerank,memory,P@5,R@5,RR
0,Q1,fixed,False,False,0.4,0.222,1.000
1,Q2,fixed,False,False,1.0,0.333,1.000
2,Q3,fixed,False,False,1.0,0.081,1.000
3,T1,fixed,False,False,1.0,0.250,1.000
4,T2,fixed,False,False,0.0,0.000,0.000
5,T2,fixed,False,True,0.2,0.143,1.000
6,Q1,fixed,True,False,0.8,0.444,1.000
7,Q2,fixed,True,False,1.0,0.333,1.000
8,Q3,fixed,True,False,1.0,0.081,1.000
9,T1,fixed,True,False,1.0,0.250,1.000


In [8]:
m = ["P@5", "R@5", "RR"]  # mean RR = MRR
base = df[df["query"] != "T2"]
display(base.groupby("rerank")[m].mean().round(3))
display(base.groupby("strategy")[m].mean().round(3))
display(df[df["query"] == "T2"].groupby("memory")[m].mean().round(3))

,P@5,R@5,RR
rerank,,,
False,0.875,0.259,0.938
True,0.950,0.304,1.000


,P@5,R@5,RR
strategy,,,
fixed,0.900,0.249,1.000
sentence,0.925,0.314,0.938


,P@5,R@5,RR
memory,,,
False,0.05,0.042,0.083
True,0.35,0.274,1.000


## Findings
| Comparison | P@5 | R@5 | MRR |
|---|---|---|---|
| Without reranking | 0.875 | 0.259 | 0.938 |
| With reranking | 0.950 | 0.304 | 1.000 |
| Fixed chunks | 0.900 | 0.249 | 1.000 |
| Sentence chunks | 0.925 | 0.314 | 0.938 |
| Follow up without memory | 0.050 | 0.042 | 0.083 |
| Follow up with memory | 0.350 | 0.274 | 1.000 |

* **Reranking**: all of the gain comes from Q1. Precision rose from 0.4 to 0.8 with fixed chunks and from 0.6 to 0.8 with sentence chunks, and with sentence chunks the cross encoder moved a relevant chunk from rank 2 to rank 1. That one move is the whole MRR gain (0.938 to 1.000). Q2, Q3 and T1 were already at P@5 = 1.0 and stayed there.
* **Chunking**: sentence chunks show higher R@5, but mostly because the sentence collection has fewer relevant chunks (no overlap duplicates): 7 vs 9 for Q1, 13 vs 15 for Q2, 53 vs 62 for Q3, 18 vs 20 for T1. The same 5 relevant hits are then a larger fraction. Q1 is the only query where sentence chunks retrieved more relevant chunks (3 vs 2 without reranking), which gives them the slightly higher P@5, while the fixed chunks put a relevant chunk first more often (higher MRR). R@5 can be at most 5 divided by the number of relevant chunks (0.081 for Q3 fixed), so recall at K = 5 says little here.
* **Memory** had the largest effect. Alone, "How was its temperature determined?" retrieves water, accretion disk and brown dwarf papers. With the previous turn added, all top 5 chunks come from the white dwarf paper. Precision stays at 0.35 because the combined query is dominated by the first turn, so some chunks discuss abundances rather than temperature. Rewriting the follow up to replace the pronoun with the topic, or weighting the current turn more heavily, would likely help.
* With 4 standalone queries, 1 follow up and keyword based chunk labels, these differences are indicative, not conclusive.